# 手撕 ALiBi (Attention with Linear Biases)

## 背景
ALiBi 不使用位置编码，而是在 attention score 上加一个线性递减 bias。
bias[i][j] = -(i - j) * slope[head]，其中 slope 按几何级数生成。
优势：外推性好，无需位置编码嵌入。

## 考察点
- ALiBi bias 矩阵的构造
- slope 的几何级数生成方式
- 与 RoPE 的对比（ALiBi 外推性更好）

In [ ]:
import torch
import torch.nn.functional as F
import math

def get_alibi_slopes(n_heads: int) -> torch.Tensor:
    # 几何级数: 2^(-8/n) 对 n 个头
    m = 2 ** (-8 / n_heads)
    return torch.tensor([m ** i for i in range(n_heads)])

def alibi_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, slopes: torch.Tensor) -> torch.Tensor:
    # q,k,v: (batch, n_heads, seq_len, d_head)
    # slopes: (n_heads,)
    batch, n_heads, seq_len, d_head = q.shape
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_head)
    # 构造 bias: (seq_len, seq_len)
    positions = torch.arange(seq_len)
    relative = positions.unsqueeze(0) - positions.unsqueeze(1)  # (i - j)
    # bias[head, i, j] = -relative[i,j] * slope[head]
    bias = -relative.float().unsqueeze(0) * slopes.unsqueeze(1).unsqueeze(2)  # (n_heads, seq_len, seq_len)
    scores = scores + bias.unsqueeze(0)  # broadcast batch
    attn = F.softmax(scores, dim=-1)
    return attn @ v

In [ ]:
# 验证 ALiBi attention
torch.manual_seed(42)
batch, n_heads, seq_len, d_head = 2, 4, 8, 16
q = torch.randn(batch, n_heads, seq_len, d_head)
k = torch.randn(batch, n_heads, seq_len, d_head)
v = torch.randn(batch, n_heads, seq_len, d_head)
slopes = get_alibi_slopes(n_heads)
out = alibi_attention(q, k, v, slopes)
assert out.shape == (batch, n_heads, seq_len, d_head)
# 验证 bias 矩阵性质：对角线为 0，上方为正，下方为负
positions = torch.arange(seq_len)
relative = positions.unsqueeze(0) - positions.unsqueeze(1)
bias_head0 = -relative.float() * slopes[0]
assert bias_head0[0, 0] == 0, "对角线 bias=0"
assert bias_head0[1, 0] < 0, "下三角 bias<0（未来位置惩罚）"
print(f"slopes: {slopes.tolist()}")
print(f"output shape: {out.shape}")
print("✅ ALiBi attention 验证通过")